# Lie Groups: SO(2) and SO(3) and their Representations

**6.7970/8.750 Symmetry and its Application to Machine Learning**

Companion notebook for the [Lie Groups notes](https://symm4ml.mit.edu/symm4ml_s26/notes/lie-groups). We will:

1. Review the generators of SO(2) and SO(3)
2. Show that the SO(2) vector rep is reducible (diagonalize the generator)
3. Show that the SO(3) vector rep is irreducible
4. Build higher irreps via tensor products (`infer_irreps_from_tensor_products`)
5. Examine the $l=2$ generators and verify the Casimir $L^2 = -l(l+1)\mathbb{1}$
6. Compute CG coefficients from tensor product decomposition

In [ ]:
%%capture
!pip install torch
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [ ]:
import numpy as np
from symm4ml import vis, linalg, lie, so3

---
## 1. Generators of SO(2)

SO(2) has a single generator (one plane of rotation):

In [ ]:
so2_generator = np.array([
    [0, -1],
    [1,  0]
], dtype=float)

print('SO(2) generator L:')
print(so2_generator)
print(f'\nSkew-symmetric? {np.allclose(so2_generator, -so2_generator.T)}')
print(f'n(n-1)/2 = 2(1)/2 = 1 generator ✓')

### The SO(2) vector rep is reducible

We can diagonalize the generator with a similarity transform $U$:

In [ ]:
eigvals, U = np.linalg.eig(so2_generator)
print('Eigenvalues:', eigvals)
print('\nU (change of basis):')
print(np.round(U, 4))

# Diagonalized generator
L_diag = np.linalg.inv(U) @ so2_generator @ U
print('\nU^{-1} L U (diagonalized):')
print(np.round(L_diag, 4))
print('\n→ Two 1D irreps: e^{+iα} and e^{-iα}')

### SO(2) irreps via tensor products

Since all irreps are 1D, the Kronecker sum is just ordinary addition of generators:

In [ ]:
# The two 1D irrep generators
g_plus = L_diag[0, 0]   # +i
g_minus = L_diag[1, 1]  # -i

print('Irrep generators (1D):')
print(f'  m=+1: generator = {g_plus}')
print(f'  m=-1: generator = {g_minus}')

# Tensor products (Kronecker sum = ordinary sum for 1D)
print('\nTensor products (generators add):')
print(f'  m=+1 ⊗ m=+1: {g_plus} + {g_plus} = {g_plus + g_plus}  → m=+2')
print(f'  m=-1 ⊗ m=-1: {g_minus} + {g_minus} = {g_minus + g_minus}  → m=-2')
print(f'  m=+1 ⊗ m=-1: {g_plus} + {g_minus} = {g_plus + g_minus}  → m=0 (trivial)')
print('\nSelection rule: m₁ ⊗ m₂ = m₁ + m₂')

---
## 2. Generators of SO(3)

SO(3) has 3 generators (3 planes of rotation in 3D):

In [ ]:
Lx = np.array([[0,0,0],[0,0,-1],[0,1,0]], dtype=float)
Ly = np.array([[0,0,1],[0,0,0],[-1,0,0]], dtype=float)
Lz = np.array([[0,-1,0],[1,0,0],[0,0,0]], dtype=float)

so3_generators = np.stack([Lx, Ly, Lz])

print('Lx (rotation in yz-plane):')
print(Lx)
print('\nLy (rotation in xz-plane):')
print(Ly)
print('\nLz (rotation in xy-plane):')
print(Lz)

# Verify skew-symmetric
print(f'\nAll skew-symmetric? {all(np.allclose(L, -L.T) for L in [Lx, Ly, Lz])}')

### Verify the Lie algebra: $[L_i, L_j] = \epsilon_{ijk} L_k$

In [ ]:
# Build the Lie algebra (structure constants)
algebra = np.zeros([3, 3, 3])
algebra[0, 1, 2] = 1   # [Lx, Ly] = Lz
algebra[1, 2, 0] = 1   # [Ly, Lz] = Lx
algebra[2, 0, 1] = 1   # [Lz, Lx] = Ly
algebra[1, 0, 2] = -1  # [Ly, Lx] = -Lz
algebra[2, 1, 0] = -1  # [Lz, Ly] = -Lx
algebra[0, 2, 1] = -1  # [Lx, Lz] = -Ly

print('Is a representation of the Lie algebra?', 
      lie.is_a_representation(algebra, so3_generators))

# Show one commutator explicitly
print('\n[Lx, Ly] =')
print(Lx @ Ly - Ly @ Lx)
print('\nLz =')
print(Lz)
print('\nEqual?', np.allclose(Lx @ Ly - Ly @ Lx, Lz))

### The SO(3) vector rep is irreducible

Unlike SO(2), no single similarity transform can simultaneously block-diagonalize all three generators:

In [ ]:
from scipy.linalg import expm
import matplotlib.pyplot as plt

# Each generator individually has a 2x2 block + a fixed axis
print('e^(α Lx) — rotation in yz-plane, x fixed:')
alpha = 0.5
print(np.round(expm(alpha * Lx), 3))

print('\ne^(α Ly) — rotation in xz-plane, y fixed:')
print(np.round(expm(alpha * Ly), 3))

print('\ne^(α Lz) — rotation in xy-plane, z fixed:')
print(np.round(expm(alpha * Lz), 3))

print('\nThe "block" structure does NOT align across generators → irreducible!')

---
## 3. Building Irreps via Tensor Products

Use `infer_irreps_from_tensor_products` to systematically discover all SO(3) irreps:

In [ ]:
# Get first 5 irreps (l = 0, 1, 2, 3, 4)
irreps = lie.infer_irreps_from_tensor_products(so3_generators, n=5)

print(f'Found {len(irreps)} irreps:')
for i, ir in enumerate(irreps):
    dim = ir.shape[1]
    L2_val = (ir @ ir).sum(0)[0, 0]
    l = i  # irreps come out in order l=0,1,2,...
    print(f'  l={l}: dim = {dim} (= 2l+1 = {2*l+1}), L² = {L2_val:.1f} (= -l(l+1) = {-l*(l+1)})')

### Visualize the generators for each irrep

In [ ]:
for i, ir in enumerate(irreps):
    print(f'\n=== l = {i} (dim {ir.shape[1]}) ===')
    vis.plot_matrices(ir, figsize=(5, 2), cmap='RdBu', vmax=np.abs(ir).max(), vmin=-np.abs(ir).max());
    plt.show()

---
## 4. The $l=2$ Generators in Detail

The $l=2$ irrep has $5 \times 5$ skew-symmetric generators, obtained from the tensor product $l=1 \otimes l=1$:

In [ ]:
l2_gens = irreps[2]  # shape (3, 5, 5)
Lx2, Ly2, Lz2 = l2_gens

print('l=2 Lx:')
print(np.round(Lx2, 3))
print('\nl=2 Ly:')
print(np.round(Ly2, 3))
print('\nl=2 Lz:')
print(np.round(Lz2, 3))

# Verify same Lie algebra
print(f'\nSame Lie algebra? {lie.is_a_representation(algebra, l2_gens)}')
print(f'All skew-symmetric? {all(np.allclose(L, -L.T) for L in l2_gens)}')

### The Casimir operator $L^2 = L_x^2 + L_y^2 + L_z^2$

Each $L_i^2$ individually is NOT proportional to $\mathbb{1}$, but the sum is (Schur's lemma):

In [ ]:
for l_idx in [1, 2, 3]:
    gen = irreps[l_idx]
    Lx_l, Ly_l, Lz_l = gen
    
    print(f'\n=== l = {l_idx} (dim {gen.shape[1]}) ===')
    print('Lx²:')
    print(np.round(Lx_l @ Lx_l, 2))
    print('Ly²:')
    print(np.round(Ly_l @ Ly_l, 2))
    print('Lz²:')
    print(np.round(Lz_l @ Lz_l, 2))
    
    L2 = Lx_l @ Lx_l + Ly_l @ Ly_l + Lz_l @ Lz_l
    print(f'\nL² = Lx² + Ly² + Lz²:')
    print(np.round(L2, 2))
    print(f'= {L2[0,0]:.1f} × I_{gen.shape[1]}  (expected: {-l_idx*(l_idx+1)} × I)')

---
## 5. Tensor Product Decomposition and Selection Rules

### $l=1 \otimes l=1 = l=0 \oplus l=1 \oplus l=2$

In [ ]:
# Tensor product of l=1 with itself (Kronecker sum)
vec_vec = lie.tensor_product(so3_generators, so3_generators)
print(f'Tensor product generators shape: {vec_vec.shape}')  # (3, 9, 9)

# Decompose into irreps
decomposed = lie.decompose_rep_into_irreps(vec_vec)
print(f'\nDecomposition: {len(decomposed)} irreps')
for ir in decomposed:
    dim = ir.shape[1]
    L2_val = (ir @ ir).sum(0)[0, 0]
    l = round((-1 + np.sqrt(1 - 4*L2_val)) / 2)
    print(f'  l={l}: dim {dim}')
    vis.plot_matrices(ir, figsize=(5, 2), cmap='RdBu', vmax=np.abs(ir).max(), vmin=-np.abs(ir).max());
    plt.show()

### Selection rule (triangle inequality)

$l_1 \otimes l_2 = |l_1 - l_2| \oplus (|l_1 - l_2| + 1) \oplus \cdots \oplus (l_1 + l_2)$

Equivalently: $l_3$ appears iff $|l_1 - l_2| \leq l_3 \leq l_1 + l_2$

In [ ]:
# Verify selection rule for several tensor products
for l1, l2 in [(1, 1), (1, 2), (1, 3), (2, 2)]:
    tp = lie.tensor_product(irreps[l1], irreps[l2])
    decomposed = lie.decompose_rep_into_irreps(tp)
    l_values = []
    for ir in decomposed:
        L2_val = (ir @ ir).sum(0)[0, 0]
        l = round((-1 + np.sqrt(1 - 4*L2_val)) / 2)
        l_values.append(l)
    expected = list(range(abs(l1 - l2), l1 + l2 + 1))
    print(f'l={l1} ⊗ l={l2} = {" ⊕ ".join(f"l={l}" for l in sorted(l_values))}')
    print(f'  Expected (triangle): {" ⊕ ".join(f"l={l}" for l in expected)}')
    print(f'  Match? {sorted(l_values) == expected}')
    print()

---
## 6. Clebsch-Gordan Coefficients

The CG coefficients are the change-of-basis matrices from tensor product to irreps — exactly `infer_change_of_basis` applied to the tensor product generators:

In [ ]:
# CG for l=1 ⊗ l=1 → l=2
vec_vec = lie.tensor_product(so3_generators, so3_generators)

# Find how l=2 embeds in the tensor product
cob = linalg.infer_change_of_basis(vec_vec, irreps[2])
print(f'Change of basis shape: {cob.shape}')  # (n_solutions, 9, 5)
print(f'  = ({cob.shape[0]} solutions, {cob.shape[1]} tensor product dim, {cob.shape[2]} irrep dim)')

# Reshape to see CG coefficients as (l1_components × l2_components) → l3_components
cg = cob[0].reshape(3, 3, 5)  # (l1=1 dim, l2=1 dim, l3=2 dim)
print(f'\nCG coefficients reshaped: {cg.shape}')
print('These are the coefficients for combining two vectors into a rank-2 tensor')

In [ ]:
# Visualize the 5 CG coefficient matrices (one per l=2 component)
# Each shows how the 3×3 outer product of two vectors projects onto one l=2 component
fig = vis.plot_matrices(np.einsum('ijk->kij', cg.real), cmap='RdBu')
plt.suptitle('CG coefficients: l=1 ⊗ l=1 → l=2\n(5 matrices, one per l=2 component)', y=1.05)
plt.show()

print('These are the 5 symmetric traceless patterns — the spherical harmonics Y₂ᵐ')
print('evaluated on the grid {xx, xy, xz, yx, yy, yz, zx, zy, zz}')

---
## Summary

| | SO(2) | SO(3) |
|---|---|---|
| **Generators** | 1 (skew-symmetric 2×2) | 3 (skew-symmetric 3×3) |
| **Lie algebra** | $[L, L] = 0$ (trivial) | $[L_i, L_j] = \epsilon_{ijk} L_k$ |
| **Vector rep** | Reducible (→ $e^{i\alpha} \oplus e^{-i\alpha}$) | Irreducible ($l = 1$) |
| **Irreps** | 1D: $e^{im\alpha}$, $m \in \mathbb{Z}$ | $(2l+1)$-dim, $l = 0, 1, 2, \ldots$ |
| **Casimir** | $L^2 = -m^2$ | $L^2 = -l(l+1) \cdot \mathbb{1}$ |
| **Selection rule** | $m_1 \otimes m_2 = m_1 + m_2$ | $\lvert l_1 - l_2 \rvert \leq l_3 \leq l_1 + l_2$ |
| **CG coefficients** | Trivial (1D) | → Spherical harmonics |